### Use LangChain's built-in tools/ creating custom tools and understand how tools are called

In [1]:
import langchain
import langchain_core
import langchain_openai
import langchain_community
from importlib.metadata import version
import transformers
import numpy as np

print(version("langchain"))
print(version("langchain-core"))
print(version("langchain-openai"))
print(version("langchain-community"))

print("langchain_v",langchain.__version__)
print("langchain_core",langchain_core.__version__)
print("langchain_community",langchain_community.__version__)
print("transformers",transformers.__version__)
print("numpy",np.__version__)

1.2.3
1.4.8
1.1.7
0.4.1
langchain_v 1.2.3
langchain_core 1.4.8
langchain_community 0.4.1
transformers 4.55.2
numpy 1.26.4


In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
#!pip install wikipedia

In [3]:
!pip list | FINDSTR wikipedia

wikipedia                                1.4.0


In [3]:
# ------------------------------------------------------------
# Step 1: Initialize Wikipedia tool
# ------------------------------------------------------------
#WikipediaQueryRun already inherits from BaseTool
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

In [9]:
result = wikipedia_tool.invoke("Tom cruise")
print(result)

Page: Tom Cruise
Summary: Thomas Cruise  Mapother IV ( MAY-poth-ər; born July 3, 1962) is an American actor and film producer. Regarded as a Hollywood icon, he has received various accolades, including an Honorary Palme d'Or, an Academy Honorary Award, and three Golden Globes, in addition to nominations for four competitive Academy Awards. As of 2026, his films have grossed more than $13.3 billion worldwide, placing him among the highest-grossing actors of all time. One of Hollywood's most bankable stars, he is consistently one of the world's highest-paid actors.
Cruise began acting in the early 1980s and made his breakthrough with leading roles in Risky Business (1983) and Top Gun (1986), the latter earning him a reputation as a sex symbol. Critical acclaim came with his roles in the dramas The Color of Money (1986), Rain Man (1988), and Born on the Fourth of July (1989). For his portrayal of Ron Kovic in the latter, he won a Golden Globe Award and received a nomination for the Academ

In [4]:
from langchain_core.tools import Tool

calculator_tool = Tool(
    name="Calculator",
    func=lambda x: str(eval(x)),
    description="Useful for running math expressions like '3+12'"
)

from math import sqrt 
#Using pre-defined function
def calc(expression: str) -> str:
    return str(eval(expression))

calculator_tool2 = Tool(
    name="Calculator",
    func=calc,
    description="Evaluate math expressions"
)

In [11]:
result = calculator_tool.invoke("3 + 12")
print(result)

15


In [12]:
#Or LangChain tools also support .run()
result = calculator_tool.run("10 * 25")
print(result)

250


In [13]:
result = calculator_tool2.invoke("sqrt(16) + 5")
print(result)

9.0


In [5]:
#Other option i.e. creating a custom tool
from langchain.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper

wiki_api = WikipediaAPIWrapper()

@tool
def wikipedia_search(query: str) -> str:
    """Useful for fetching facts from Wikipedia."""
    return wiki_api.run(query)
#uncomment below to test
#wikipedia_search.invoke("Marie Curie")

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))
#uncomment below to test
#calculator.invoke("3 + 4 * 2")

from langchain.tools import tool
import math

@tool
def calculator2(expression: str) -> str:
    """Evaluate mathematical expressions safely."""
    allowed_names = {
        "abs": abs,
        "round": round,
        "sqrt": math.sqrt,
        "pow": pow,
        "pi": math.pi,
        "e": math.e,
    }
    return str(eval(expression, {"__builtins__": {}}, allowed_names))
#uncomment below to test   
#calculator2.invoke("3 + 4 * 2")

e:\Lesson_2_demos\venv\lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [70]:
#Or
'''
print("=== Wikipedia Tool ===")
print(wikipedia_search.invoke("Marie Curie"))

print("\n=== Calculator Tool ===")
print(calculator.invoke("3 + 4 * 2"))

print("\n=== Safe Calculator Tool ===")
print(calculator2.invoke("sqrt(81) + pi"))
'''

'\nprint("=== Wikipedia Tool ===")\nprint(wikipedia_search.invoke("Marie Curie"))\n\nprint("\n=== Calculator Tool ===")\nprint(calculator.invoke("3 + 4 * 2"))\n\nprint("\n=== Safe Calculator Tool ===")\nprint(calculator2.invoke("sqrt(81) + pi"))\n'

### Build a simple agent using a prompt template + 2 tools (e.g., calculator + search)

In [6]:
from langchain_huggingface import HuggingFacePipeline
from langchain_community.agent_toolkits.load_tools import load_tools as lc_load_tools
from transformers import pipeline
from langchain_core.runnables import RunnableSequence

In [8]:
import langchain
import langchain_core
import langchain_openai
import langchain_community
from importlib.metadata import version
import transformers
import numpy as np

print("langchain_v",langchain.__version__)
print("langchain_core",langchain_core.__version__)
print("langchain_community",langchain_community.__version__)
print("transformers",transformers.__version__)
print("numpy",np.__version__)

langchain_v 1.2.3
langchain_core 1.4.8
langchain_community 0.4.1
transformers 4.55.2
numpy 1.26.4


In [9]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer



In [12]:
# hf_pipeline = pipeline(
#     "text2text-generation",
#     model="google/flan-t5-large",
#     max_new_tokens=256,
# )

model_name = "google/flan-t5-large"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    local_files_only=True
)

hf_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
)


llm = HuggingFacePipeline(pipeline=hf_pipeline)

Device set to use cpu


In [22]:
#If using bigger LLMs
import os
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
#load_dotenv("/content/.env")
load_dotenv()

llm2 = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    deployment_name=os.getenv("AZURE_DEPLOYMENT_NAME"),
    temperature=0,
)

#To use existing (built-in)tools
#!pip install numexpr
tools = lc_load_tools(["wikipedia", "llm-math"], llm=llm2)

#we can also do to add our custom tools
#tools = lc_load_tools(["wikipedia", "llm-math"], llm=llm) + [calculator2]

In [27]:
#llm2.invoke('explain quantum physics')

In [14]:
type(tools)

list

In [23]:
for t in tools:
    print(t.name, "->", t.description)

wikipedia -> A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
Calculator -> Useful for when you need to answer questions about math.


In [28]:
#Building simple agent
def get_tool_by_keyword(keyword):
    for t in tools:
        if keyword.lower() in t.name.lower():
            return t
    return None

def agent(query: str):
    q = query.lower()

    # Math routing
    if any(x in q for x in ["*", "+", "-", "/", "calculate", "math"]):
        tool = get_tool_by_keyword("math") or get_tool_by_keyword("calc")
        if tool:
            return tool.run(query)

    # Wikipedia routing
    if any(x in q for x in ["who", "what", "where", "capital", "wiki"]):
        tool = get_tool_by_keyword("wikipedia")
        if tool:
            return tool.run(query)

    # Fallback
    return llm2.invoke(query)

In [32]:
#agent("What is the capital of France?")
agent("calculate 10**9")

'Answer: 1000000000'

In [34]:
#Using tools, llm and templates
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = "Please write a {length} review of the book {book_title}."
prompt = PromptTemplate(
    input_variables=["length", "book_title"],
    template=template
)

llm_chain = prompt | llm2 | StrOutputParser()

In [35]:
#Approach 1: Tool-first, then LLM
#Use tools when the query clearly needs factual information. 
# Then pass the gathered facts to the LLM for final writing.
'''
This is ideal when:

factual accuracy matters
you need external information
the LLM should synthesize rather than invent
'''
def review_with_tools_first(book_title: str, length: str = "short"):
    # Fetch facts from Wikipedia
    wiki_info = wikipedia_search.invoke(book_title)

    enhanced_prompt = f"""
    Using the following factual information:

    {wiki_info}

    Write a {length} review of the book {book_title}.
    Include a brief summary, themes, and overall impression.
    """

    return llm2.invoke(enhanced_prompt)

In [36]:
print(review_with_tools_first("The Hobbit", "short"))

content='**Review of The Hobbit**\n\nJ.R.R. Tolkien’s *The Hobbit* is a timeless classic of children’s fantasy literature, first published in 1937 and still beloved today. The story follows Bilbo Baggins, a home-loving hobbit from the peaceful town of Hobbiton, who is swept into an adventurous quest by the wizard Gandalf and thirteen dwarves. Their mission: to reclaim the dwarves’ lost home and treasure from the fearsome dragon Smaug. Along the way, Bilbo encounters a series of dangers and monsters, each episode testing his courage and wit.\n\nThe novel is structured as an episodic quest, with each chapter introducing new challenges and characters. Central themes include personal growth, heroism, and the acceptance of adventure and change. Bilbo’s journey is as much about self-discovery as it is about treasure, and his transformation from a timid hobbit to a clever and brave hero is both compelling and inspiring. The story also explores motifs of warfare, friendship, and the struggle b

In [37]:
#Approach 2: LLM-first, then Tools if Needed
#Let the LLM answer first. If the response is weak, incomplete, or uncertain, then call tools.
'''
This is ideal when:

many questions can be answered directly
you want to minimize tool usage
tools are expensive or slow
'''
from langchain_core.tools import tool
import wikipedia

@tool
def wikipedia_search(query: str) -> str:
    """Fetch a short summary from Wikipedia for a given topic."""
    try:
        return wikipedia.summary(query, sentences=5)
    except Exception as e:
        return f"Wiki lookup failed: {e}"

def review_with_llm_first(book_title: str, length: str = "short"):
    initial_review = llm_chain.invoke({
        "length": length,
        "book_title": book_title
    })

    needs_enrichment = len(initial_review.split()) < 30

    if needs_enrichment:
        wiki_info = wikipedia_search.invoke(book_title)

        enhanced_prompt = f"""
        Improve this review:

        Review:
        {initial_review}

        Facts:
        {wiki_info}

        Write a {length} review of {book_title}.
        """

        return llm2.invoke(enhanced_prompt)

    return initial_review

In [38]:
print(review_with_llm_first("The Hobbit", "short"))

**The Hobbit** by J.R.R. Tolkien is a classic fantasy novel that follows the journey of Bilbo Baggins, a reluctant hobbit who is swept into an epic quest to reclaim a lost dwarven kingdom from the fearsome dragon Smaug. Tolkien’s storytelling is rich and imaginative, filled with memorable characters, clever riddles, and a sense of adventure that appeals to readers of all ages. The world-building is immersive, and the narrative balances humor, suspense, and heartwarming moments. As a prelude to *The Lord of the Rings*, *The Hobbit* stands on its own as a charming and timeless tale about courage, friendship, and the unexpected heroism found in ordinary people.


In [39]:
#Router pattern
def book_review_agent(book_title: str, length: str = "short", use_tools=True):
    if use_tools:
        return review_with_tools_first(book_title, length)
    else:
        return review_with_llm_first(book_title, length)

In [40]:
print(book_review_agent("The Hobbit", "short", use_tools=True))

content='**The Hobbit** by J.R.R. Tolkien is a classic fantasy novel that follows the journey of Bilbo Baggins, a comfort-loving hobbit who is swept into an epic quest by the wizard Gandalf and a group of dwarves. Their mission is to reclaim the dwarves’ homeland and treasure from the fearsome dragon, Smaug. Along the way, Bilbo encounters trolls, goblins, elves, and the mysterious creature Gollum, from whom he acquires the magical One Ring.\n\nThe novel explores themes of bravery, personal growth, and the importance of friendship. Bilbo’s transformation from a reluctant adventurer to a clever and courageous hero is at the heart of the story. Tolkien also weaves in themes of greed, hospitality, and the value of home.\n\nOverall, *The Hobbit* is a charming and imaginative tale, filled with memorable characters and vivid landscapes. Its accessible style and sense of wonder make it appealing to readers of all ages. The book’s blend of adventure, humor, and heart has cemented its place as 